# 04 Message passing

**Question.** How do M1/M2/M3 compare across poolings and splits, and what do the WL ablations store?

In [ ]:
from pathlib import Path
import os, json, csv
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "chapter_a").is_dir():
    for cand in (Path(".."), Path("../.."), Path("../../..")):
        if (cand.resolve() / "chapter_a").is_dir():
            ROOT = cand.resolve()
            break
os.chdir(ROOT)
MASTER = pd.read_csv(ROOT / "chapter_a" / "MASTER_RESULTS.csv")
ANDROCT = ROOT / "abrg" / "output" / "androct_2017"

def row_eq(mask, artifact_auc):
    sub = MASTER.loc[mask]
    assert len(sub) >= 1, mask
    mval = float(sub.iloc[0]["auc_floor"])
    aval = float(artifact_auc)
    assert round(mval, 6) == round(aval, 6), (mval, aval)


In [ ]:
print(pd.read_csv(ROOT / "chapter_a" / "tables" / "T5_message_passing.csv").to_string(index=False))
ab = json.loads((ANDROCT / "kernels" / "ablation" / "winner_ablation.json").read_text())
row_eq(MASTER.detector=="WL_edges_removed", ab["edges_removed_auc_floor"])
row_eq(MASTER.detector=="WL_structure_only_features_constant", ab["features_constant_auc_floor"])
sa = json.loads((ANDROCT / "supgnn" / "splitA" / "splitA_results.json").read_text())
floors = [s["auc"]["auc_floor"] for s in sa["T22"]["mean"]["M1_full"]["per_seed"]]
mean = float(np.mean(floors))
row_eq((MASTER.representation=="T22_mean") & (MASTER.method=="M1_full") & (MASTER.split=="splitA_stratified"), mean)
print("ok")
